## Support tickets — dataloader test

This notebook checks that **`supportDataset`** and **`DataLoader`** work for fine-tuning on support-ticket text.

- **CSV**: Each row has a ticket `text` and a string `label` (e.g. `shipping_delay`). The dataset maps those strings to integer class ids `0 … num_classes-1`.
- **Tokenizer**: GPT-2 byte-pair encoding via `tiktoken`; every example is padded or truncated to a fixed length (`max_length=None` uses the longest sequence in the split).
- **Sampling**: `sample_from_dataset` shows individual examples; `peek_dataloader_batch` shows one full batch after stacking (default PyTorch collate).

The **next cell** summarizes what each printed line means. **After that**, run the code cell to load the data and print samples.

### What you should see in the output

| Print | Meaning |
|--------|---------|
| `7` | Number of distinct ticket categories (`num_classes`). |
| `torch.Size([12])`, `4`, `shipping_delay` | One example: token ids length 12 (padded length for this split), class id `4`, and the original label name. Three such lines = three random rows. |
| `torch.Size([8, 12])`, `torch.Size([8])` | One batch: **8** sequences of length **12**, and **8** integer labels—this is the tensor shape the training loop will consume. |

In [10]:
import tiktoken
from torch.utils.data import DataLoader
from gpt_class_finetune import SupportDataset, sample_from_dataset, peek_dataloader_batch

# Initialize tokenizer
tokenizer = tiktoken.get_encoding('gpt2')

# Initialize dataset
train_dataset = SupportDataset(
    csv_file='data/support_tickets_train.csv',
    tokenizer=tokenizer,
    max_length=None,
    balance_dataset=True
)

print(train_dataset.num_classes)

# Create DataLoader
num_workers = 0
batch_size = 8

train_loader = DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, drop_last=False
)

rows = sample_from_dataset(train_dataset, k=3, seed=43)
for input_ids, y in rows:
    print(input_ids.shape, y.item(), train_dataset.id_to_label[y.item()])

inputs, labels = peek_dataloader_batch(train_loader)
print(inputs.shape, labels.shape)


7
torch.Size([12]) 3 password_reset
torch.Size([12]) 2 cancel_subscription
torch.Size([12]) 5 technical_bug
torch.Size([8, 12]) torch.Size([8])


In [11]:
import pandas as pd

df = pd.read_csv('data/support_tickets_train.csv')
vc = df['label'].value_counts()
spread_pct = (vc.max() / vc.min() - 1) * 100
print(f'Before balancing: Max class count is {spread_pct:.1f}% above the min class count')
print(df['label'].value_counts())

print('\n')

balanced_df = train_dataset._create_balanced_dataset()
print('After balancing:')
print(balanced_df['label'].value_counts())


Before balancing: Max class count is 15.2% above the min class count
label
billing_refund         167
account_locked         157
technical_bug          156
cancel_subscription    155
update_payment         153
password_reset         145
shipping_delay         145
Name: count, dtype: int64


After balancing:
label
billing_refund         145
account_locked         145
technical_bug          145
cancel_subscription    145
update_payment         145
password_reset         145
shipping_delay         145
Name: count, dtype: int64
